# 구위 모델 v3 정식화 (H5★ = F + HAVAA) · 예비 v3_h4 (H4★ = v3 + 보정 HAA)

근거 : `stuff_pipeline_261010.ipynb` (1·2차), `_vaa3`·`_vaa4`·`_vaa5` (3~5차). 웹 반영 모델은 **v3** (사용자 지시 2026-10-10). v3_h4 는 준비만 (`export_web_data.MODEL_VERSION` 을 바꾸면 사용)

- 3단계 피처 (모델 입력) : 구속·무브먼트(pfx)·릴리스 위치·익스텐션·회전·팔각도·주 패스트볼 대비 차이·구종·투구손 + **HAVAA**
  - 궤적 초기값·가속도(vx0·vy0·vz0·ax·ay·az)와 원값 VAA·HAA 는 투구 위치를 담고 있어 제외
- HAVAA = vaa − (a + b·plate_z), 구종별 선형회귀 잔차 (2025 정규시즌 학습기간 적합) — Chamberlain 'VAA Above Average'
- 보정 계수는 모델 객체(`stuff_model.AngleAdjustedStuffModel`)에 함께 저장 → export·SHAP·daily 는 원값 vaa·plate_z 를 넘기면 된다
- LGBM 은 `outputs/vaa_exp/stage3_H5t.joblib` / `stage3_H4t.joblib` (F 와 같은 전체 튜닝, sp.fit_stuff_model n_iter=12)
- 스케일러 : 학습기간 점수 대상(waste 볼 제외) 투구 예측으로 sp.StuffScaler 적합 (v2 와 같은 규칙)

In [ ]:
import os, json, joblib
import numpy as np
import pandas as pd
import stuff_pipeline as sp
import stuff_model as sm

OUT = os.path.join('outputs', 'vaa_exp')
CACHE = os.path.join(OUT, 'cache')
F_FEATS = [f for f in sp.STUFF_FEATURES if f not in ('vx0', 'vz0', 'ax', 'ay', 'az', 'vy0')]
cols = sorted(set(F_FEATS) | {'game_date', 'vaa', 'haa', 'plate_z', 'plate_x', 'waste'})
d = pd.read_parquet(os.path.join(CACHE, 'frame_2025.parquet'), columns=cols)
valid_start = pd.Timestamp(pd.read_json(os.path.join(CACHE, 'meta.json'), typ='series')['valid_start'])
is_tr = d['game_date'] < valid_start

def fit_coef(df, y, xs, keys):
    coef = {}
    for k, g in df.groupby(keys, observed=True):
        g = g[[y] + xs].astype(float).dropna()
        if len(g) >= 500:
            A = np.column_stack([np.ones(len(g))] + [g[x].to_numpy() for x in xs])
            coef[tuple(map(str, k if isinstance(k, tuple) else (k,)))] = np.linalg.lstsq(A, g[y].to_numpy(), rcond=None)[0]
    return coef

lefty = (d['p_throws'] == 'L').to_numpy()
d['haa_raw'] = np.where(lefty, -d['haa'].to_numpy('float64', na_value=np.nan), d['haa'].to_numpy('float64', na_value=np.nan))
d['rel_x_raw'] = np.where(lefty, -d['release_pos_x'].to_numpy('float64', na_value=np.nan), d['release_pos_x'].to_numpy('float64', na_value=np.nan))
vaa_coef = fit_coef(d[is_tr], 'vaa', ['plate_z'], ['pitch_type'])
haa_coef = fit_coef(d[is_tr], 'haa_raw', ['plate_x', 'rel_x_raw'], ['pitch_type', 'p_throws'])
print('HAVAA 계수 구종', len(vaa_coef), '/ 보정 HAA 계수 구종×투구손', len(haa_coef))

In [ ]:
specs = {'v3': ('stage3_H5t.joblib', F_FEATS + ['vaa_aa'], None), 'v3_h4': ('stage3_H4t.joblib', F_FEATS + ['vaa_aa', 'haa_aa'], haa_coef)}
for ver, (fname, feats, hc) in specs.items():
    lgbm = joblib.load(os.path.join(OUT, fname))
    assert list(lgbm.booster_.feature_name()) == feats, (ver, lgbm.booster_.feature_name())
    m = sm.AngleAdjustedStuffModel(lgbm, feats, vaa_coef, hc)
    X = d[m.input_features]
    # 검증 1 : 모델 객체 예측 == 실험 방식(보정 열을 직접 만들어 LGBM 예측)
    ref = d[F_FEATS].copy()
    adj = m.adjusted(d)
    for k, v in adj.items():
        ref[k] = v
    p = m.predict(X)
    assert np.array_equal(p, lgbm.predict(ref[feats])), ver
    # 검증 2 : 기여 합 = 예측 (Saabas / TreeSHAP 표본)
    for exact in (False, True):
        s = X.sample(20000, random_state=42)
        c, b = m.explain(s, exact)
        assert np.abs(b + c.sum(1) - m.predict(s)).max() < 1e-9, (ver, exact)
    scaler = sp.StuffScaler().fit(m.predict(X[is_tr & ~d['waste']]))
    out = os.path.join('models', ver)
    os.makedirs(out, exist_ok=True)
    joblib.dump(m, os.path.join(out, sm.MODEL_FILE))
    joblib.dump(scaler, os.path.join(out, 'stage4_stuff_scaler.joblib'), compress=3)
    sc = scaler.transform(p)
    print(f'{ver} : 입력 {len(m.input_features)}개, 2025 학습기간 점수 중앙값 {np.median(sc[(is_tr & ~d["waste"]).to_numpy()]):.2f}, 저장 {out}')
    json.dump({'features': feats, 'input_features': m.input_features}, open(os.path.join(out, 'features.json'), 'w'), indent=1)